In [ ]:
import pandas as pd

def get_team_avg_stats(df, team_id, season, day, n_matches=5):
    """
    Calculate the average statistics for a given team over the previous n matches
    in a specified season prior to a given day.
    
    Parameters:
    - df (pd.DataFrame): DataFrame containing game data with columns:
        Season, DayNum, WTeamID, WScore, LTeamID, LScore, WLoc, NumOT, 
        WFGM, WFGA, WFGM3, WFGA3, WFTM, WFTA, WOR, WDR, WAst, WTO, WStl, 
        WBlk, WPF, LFGM, LFGA, LFGM3, LFGA3, LFTM, LFTA, LOR, LDR, LAst, 
        LTO, LStl, LBlk, LPF.
    - team_id (int): The team's unique identifier.
    - season (int): The season in which to look for matches.
    - day (int): The day number. Only games with DayNum < day are considered.
    - n_matches (int): The number of most recent matches to average over.
    
    Returns:
    - avg_stats (pd.Series): A pandas Series containing the average for each statistic.
      If no matches are found, returns None.
    """
    # Filter for the given season and games before the specified day
    df_filtered = df[(df['Season'] == season) & (df['DayNum'] < day)]
    
    # List to collect team-specific stats from each match
    stats_list = []
    
    for _, row in df_filtered.iterrows():
        # Check if the team participated in the game
        if row['WTeamID'] == team_id:
            # Team was the winner; use winning team statistics
            stats = {
                'Score': row['WScore'],
                'FGM': row['WFGM'],
                'FGA': row['WFGA'],
                'FGM3': row['WFGM3'],
                'FGA3': row['WFGA3'],
                'FTM': row['WFTM'],
                'FTA': row['WFTA'],
                'OR': row['WOR'],
                'DR': row['WDR'],
                'Ast': row['WAst'],
                'TO': row['WTO'],
                'Stl': row['WStl'],
                'Blk': row['WBlk'],
                'PF': row['WPF']
            }
            # For the winning team, the game location is directly from WLoc
            game_loc = row['WLoc']  # 'H' (home), 'A' (away), or 'N' (neutral)
        
        elif row['LTeamID'] == team_id:
            # Team was the loser; use losing team statistics
            stats = {
                'Score': row['LScore'],
                'FGM': row['LFGM'],
                'FGA': row['LFGA'],
                'FGM3': row['LFGM3'],
                'FGA3': row['LFGA3'],
                'FTM': row['LFTM'],
                'FTA': row['LFTA'],
                'OR': row['LOR'],
                'DR': row['LDR'],
                'Ast': row['LAst'],
                'TO': row['LTO'],
                'Stl': row['LStl'],
                'Blk': row['LBlk'],
                'PF': row['LPF']
            }
            # For the losing team, the location is the opposite of the winning team's location
            if row['WLoc'] == 'H':
                game_loc = 'A'
            elif row['WLoc'] == 'A':
                game_loc = 'H'
            else:
                game_loc = 'N'
        else:
            # Skip games where the team did not participate
            continue
        
        # One-hot encoding for game location
        stats['home'] = 1 if game_loc == 'H' else 0
        stats['away'] = 1 if game_loc == 'A' else 0
        stats['neutral'] = 1 if game_loc == 'N' else 0
        
        # Include DayNum to help select the most recent matches
        stats['DayNum'] = row['DayNum']
        
        stats_list.append(stats)
    
    # If no matches are found, notify and return None
    if not stats_list:
        print(f"No matches found for team {team_id} in season {season} before day {day}.")
        return None
    
    # Create a DataFrame from the collected statistics
    stats_df = pd.DataFrame(stats_list)
    
    # Sort by DayNum descending (most recent games first)
    stats_df = stats_df.sort_values(by='DayNum', ascending=False)
    
    # Select the most recent n_matches
    recent_matches = stats_df.head(n_matches)
    
    # Compute the average for each statistic (excluding DayNum)
    avg_stats = recent_matches.drop(columns=['DayNum']).mean()
    
    return avg_stats




In [6]:
df = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")
team_id = 1101
season = 2023
day = 150
avg_stats = get_team_avg_stats(df, team_id, season, day, n_matches=5)
print(avg_stats)


Score      69.8
FGM        24.0
FGA        61.4
FGM3        5.4
FGA3       20.6
FTM        16.4
FTA        21.2
OR          8.4
DR         20.6
Ast        13.6
TO         10.0
Stl         6.8
Blk         2.0
PF         20.8
home        0.4
away        0.4
neutral     0.2
dtype: float64


In [12]:
def create_training_data(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (features X and target y) from game records.

    For each match with Season >= start_year:
      - Compute the average statistics (over the previous n_matches) for both teams 
        in the same season prior to the match.
      - Create two training examples:
           1. [winner_stats | loser_stats] with target 1 (team1 wins)
           2. [loser_stats | winner_stats] with target 0 (team1 loses)

    Parameters:
    - df (pd.DataFrame): DataFrame with game data. Expected columns include:
      Season, DayNum, WTeamID, WScore, LTeamID, LScore, WLoc, NumOT, 
      WFGM, WFGA, WFGM3, WFGA3, WFTM, WFTA, WOR, WDR, WAst, WTO, WStl, 
      WBlk, WPF, LFGM, LFGA, LFGM3, LFGA3, LFTM, LFTA, LOR, LDR, LAst, 
      LTO, LStl, LBlk, LPF.
    - start_year (int): Only matches from seasons at or after this year are considered.
    - n_matches (int): Number of previous matches to use for computing averages.

    Returns:
    - X_df (pd.DataFrame): Feature DataFrame where each row is a concatenated 
                           feature vector for two teams.
    - y_df (pd.Series): Target vector where 1 means team 1 wins, and 0 means team 1 loses.
    """
    X_rows = []
    y_rows = []
    
    # Sort the DataFrame by season and day to ensure chronological order
    df_sorted = df.sort_values(by=['Season', 'DayNum'])
    
    # Iterate over each match in the dataset (only consider matches from start_year onward)
    for _, row in df_sorted.iterrows():
        season = row['Season']
        day = row['DayNum']
        if season < start_year-n_seasons:
            continue
        
        # Extract team IDs for winner and loser
        winner_id = row['WTeamID']
        loser_id = row['LTeamID']
        
        # Calculate average stats for the winning team prior to this game
        winner_stats = get_team_avg_stats(df_sorted, winner_id, season, day, n_matches)
        # Calculate average stats for the losing team prior to this game
        loser_stats = get_team_avg_stats(df_sorted, loser_id, season, day, n_matches)
        
        # If either team doesn't have prior games, skip this match.
        if winner_stats is None or loser_stats is None:
            continue
        
        # Remove the 'DayNum' column from the computed averages only if it exists.
        if 'DayNum' in winner_stats.index:
            winner_stats = winner_stats.drop('DayNum')
        if 'DayNum' in loser_stats.index:
            loser_stats = loser_stats.drop('DayNum')
        
        # Create training sample 1: team 1 is the winner, team 2 is the loser (label 1)
        sample1 = pd.concat([winner_stats, loser_stats])
        X_rows.append(sample1)
        y_rows.append(1)
        
        # Create training sample 2: team 1 is the loser, team 2 is the winner (label 0)
        sample2 = pd.concat([loser_stats, winner_stats])
        X_rows.append(sample2)
        y_rows.append(0)
    
    # Combine the collected rows into a DataFrame and Series respectively.
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df

In [8]:
df = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")

In [13]:
# and 'get_team_avg_stats' is defined as shown earlier.
#
start_year = 2019
X, y = create_training_data(df, start_year, n_matches=5)
print(X.head())
print(y.head())

No matches found for team 1104 in season 2016 before day 11.
No matches found for team 1244 in season 2016 before day 11.
No matches found for team 1105 in season 2016 before day 11.
No matches found for team 1408 in season 2016 before day 11.
No matches found for team 1112 in season 2016 before day 11.
No matches found for team 1334 in season 2016 before day 11.
No matches found for team 1115 in season 2016 before day 11.
No matches found for team 1370 in season 2016 before day 11.
No matches found for team 1116 in season 2016 before day 11.
No matches found for team 1380 in season 2016 before day 11.
No matches found for team 1120 in season 2016 before day 11.
No matches found for team 1412 in season 2016 before day 11.
No matches found for team 1124 in season 2016 before day 11.
No matches found for team 1372 in season 2016 before day 11.
No matches found for team 1125 in season 2016 before day 11.
No matches found for team 1266 in season 2016 before day 11.
No matches found for tea

KeyboardInterrupt: 

In [16]:
import pandas as pd
from tqdm import tqdm

def create_training_data(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (features X and target y) from game records.

    For each match with Season >= start_year - n_seasons:
      - Compute the average statistics (over the previous n_matches) for both teams 
        in the same season prior to the match.
      - Create two training examples:
           1. [winner_stats | loser_stats] with target 1 (team1 wins)
           2. [loser_stats | winner_stats] with target 0 (team1 loses)

    Parameters:
    - df (pd.DataFrame): DataFrame with game data. Expected columns include:
      Season, DayNum, WTeamID, WScore, LTeamID, LScore, WLoc, NumOT, 
      WFGM, WFGA, WFGM3, WFGA3, WFTM, WFTA, WOR, WDR, WAst, WTO, WStl, 
      WBlk, WPF, LFGM, LFGA, LFGM3, LFGA3, LFTM, LFTA, LOR, LDR, LAst, 
      LTO, LStl, LBlk, LPF.
    - start_year (int): Only matches from seasons at or after (start_year - n_seasons) are considered.
    - n_matches (int): Number of previous matches to use for computing averages.
    - n_seasons (int): How many seasons before the start_year to include.

    Returns:
    - X_df (pd.DataFrame): Feature DataFrame where each row is a concatenated 
                           feature vector for two teams.
    - y_df (pd.Series): Target vector where 1 means team 1 wins, and 0 means team 1 loses.
    """
    
    X_rows = []
    y_rows = []
    
    # Sort the DataFrame by Season and DayNum to ensure chronological order
    df_sorted = df.sort_values(by=['Season', 'DayNum'])
    
    # Cache to store computed average stats for (team_id, season, day)
    avg_stats_cache = {}
    
    def get_cached_team_stats(team_id, season, day, n_matches):
        key = (team_id, season, day)
        if key not in avg_stats_cache:
            avg_stats = get_team_avg_stats(df_sorted, team_id, season, day, n_matches)
            avg_stats_cache[key] = avg_stats
        return avg_stats_cache[key]
    
    # Use tqdm to show progress through the matches
    for _, row in tqdm(df_sorted.iterrows(), total=len(df_sorted), desc="Processing matches"):
        season = row['Season']
        day = row['DayNum']
        # Only consider matches from season >= start_year - n_seasons
        if season < start_year - n_seasons:
            continue
        
        # Extract team IDs for winner and loser
        winner_id = row['WTeamID']
        loser_id = row['LTeamID']
        
        # Retrieve (or compute and cache) the average stats for both teams prior to this game
        winner_stats = get_cached_team_stats(winner_id, season, day, n_matches)
        loser_stats = get_cached_team_stats(loser_id, season, day, n_matches)
        
        # If either team doesn't have prior games, skip this match.
        if winner_stats is None or loser_stats is None:
            continue
        
        # Remove the 'DayNum' column from the computed averages only if it exists.
        if 'DayNum' in winner_stats.index:
            winner_stats = winner_stats.drop('DayNum')
        if 'DayNum' in loser_stats.index:
            loser_stats = loser_stats.drop('DayNum')
        
        # Create training sample 1: team 1 is the winner, team 2 is the loser (label 1)
        sample1 = pd.concat([winner_stats, loser_stats])
        X_rows.append(sample1)
        y_rows.append(1)
        
        # Create training sample 2: team 1 is the loser, team 2 is the winner (label 0)
        sample2 = pd.concat([loser_stats, winner_stats])
        X_rows.append(sample2)
        y_rows.append(0)
    
    # Combine the collected rows into a DataFrame and Series respectively.
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df


In [17]:
start_year = 2019
X, y = create_training_data(df, start_year, n_matches=5)
print(X.head())
print(y.head())

Processing matches:  55%|█████▍    | 64355/117748 [00:02<00:01, 28747.02it/s]

No matches found for team 1104 in season 2016 before day 11.
No matches found for team 1244 in season 2016 before day 11.
No matches found for team 1105 in season 2016 before day 11.
No matches found for team 1408 in season 2016 before day 11.
No matches found for team 1112 in season 2016 before day 11.
No matches found for team 1334 in season 2016 before day 11.
No matches found for team 1115 in season 2016 before day 11.
No matches found for team 1370 in season 2016 before day 11.
No matches found for team 1116 in season 2016 before day 11.
No matches found for team 1380 in season 2016 before day 11.
No matches found for team 1120 in season 2016 before day 11.
No matches found for team 1412 in season 2016 before day 11.
No matches found for team 1124 in season 2016 before day 11.
No matches found for team 1372 in season 2016 before day 11.
No matches found for team 1125 in season 2016 before day 11.
No matches found for team 1266 in season 2016 before day 11.
No matches found for tea

Processing matches:  58%|█████▊    | 68291/117748 [03:22<02:26, 336.57it/s]  


KeyboardInterrupt: 

In [23]:
import pandas as pd
from tqdm import tqdm

def create_long_df(df):
    """
    Convert the game-level DataFrame into a long format where each row corresponds 
    to a team’s performance in a game.
    """
    # Process winner rows
    winners = df.copy()
    winners['TeamID'] = winners['WTeamID']
    winners['Score'] = winners['WScore']
    winners['FGM']   = winners['WFGM']
    winners['FGA']   = winners['WFGA']
    winners['FGM3']  = winners['WFGM3']
    winners['FGA3']  = winners['WFGA3']
    winners['FTM']   = winners['WFTM']
    winners['FTA']   = winners['WFTA']
    winners['OR']    = winners['WOR']
    winners['DR']    = winners['WDR']
    winners['Ast']   = winners['WAst']
    winners['TO']    = winners['WTO']
    winners['Stl']   = winners['WStl']
    winners['Blk']   = winners['WBlk']
    winners['PF']    = winners['WPF']
    # For winners, location is as given
    winners['Loc']   = winners['WLoc']
    winners['is_winner'] = 1

    # Process loser rows
    losers = df.copy()
    losers['TeamID'] = losers['LTeamID']
    losers['Score'] = losers['LScore']
    losers['FGM']   = losers['LFGM']
    losers['FGA']   = losers['LFGA']
    losers['FGM3']  = losers['LFGM3']
    losers['FGA3']  = losers['LFGA3']
    losers['FTM']   = losers['LFTM']
    losers['FTA']   = losers['LFTA']
    losers['OR']    = losers['LOR']
    losers['DR']    = losers['LDR']
    losers['Ast']   = losers['LAst']
    losers['TO']    = losers['LTO']
    losers['Stl']   = losers['LStl']
    losers['Blk']   = losers['LBlk']
    losers['PF']    = losers['LPF']
    # Invert location: if winning team was at home, then loser is away and vice-versa.
    losers['Loc'] = losers['WLoc'].map(lambda x: 'A' if x=='H' else ('H' if x=='A' else 'N'))
    losers['is_winner'] = 0

    common_cols = ['Season', 'DayNum', 'TeamID', 'Score', 'FGM', 'FGA', 
                   'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 
                   'Stl', 'Blk', 'PF', 'Loc', 'is_winner']
    
    long_df = pd.concat([winners[common_cols], losers[common_cols]], ignore_index=True)
    return long_df

def compute_rolling_stats(long_df, n_matches=5):
    """
    Compute the rolling (previous n_matches) average for each statistic for each team in a season.
    A shift of 1 is applied so that the current game is not included.
    Also computes the rolling averages for one-hot encoded location indicators.
    """
    # Define the stat columns to average
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
                 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    # Create one-hot columns for location
    long_df['home']    = (long_df['Loc'] == 'H').astype(int)
    long_df['away']    = (long_df['Loc'] == 'A').astype(int)
    long_df['neutral'] = (long_df['Loc'] == 'N').astype(int)
    onehot_cols = ['home', 'away', 'neutral']
    
    # Sort by TeamID, Season, and DayNum so the rolling window is correct.
    long_df = long_df.sort_values(by=['TeamID', 'Season', 'DayNum'])
    
    # Compute rolling average for each statistic (shift to use only previous games)
    for col in stat_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    for col in onehot_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    return long_df

def create_training_data_vectorized(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (X and y) using vectorized operations.

    Process only games from seasons >= (start_year - n_seasons).
    For each game, obtain the rolling averages (computed from previous n_matches)
    for both the winner and loser, then create two samples:
      - [winner_rolling | loser_rolling] with label 1 (team 1 wins)
      - [loser_rolling | winner_rolling] with label 0 (team 1 loses)

    Returns:
    - X_df (pd.DataFrame): Each row is the concatenated rolling features for team1 and team2.
    - y_df (pd.Series): The corresponding target (1 if team1 wins, 0 otherwise).
    """
    # Filter the games to those in seasons we're interested in.
    df_filtered = df[df['Season'] >= start_year - n_seasons].copy()
    print("copy created")
    
    # Create a long-format DataFrame (one row per team per game)
    long_df = create_long_df(df_filtered)

    print("long_df created")
    
    # Compute rolling averages for each team using vectorized groupby operations.
    long_df = compute_rolling_stats(long_df, n_matches=n_matches)

    print("rolling stats computed")
    
    # We'll extract the rolling features from the long_df.
    # Choose the columns that were created by compute_rolling_stats.
    rolling_cols = [col for col in long_df.columns if col.endswith('_avg')]

    print("rolling cols selected")
    
    # Merge the rolling averages back into the original game-level DataFrame.
    # For the winner:
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    # Rename columns to indicate winner stats.
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})

    print("winner stats merged")
    
    # For the loser:
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})

    print("loser stats merged")
    
    # For the winner rolling stats:
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    # Merge winner stats back into the main DataFrame using the correct key.
    df_merged = df_filtered.merge(df_winner, left_on=['Season', 'DayNum', 'WTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    # Drop the extra 'TeamID' column from the merge.
    df_merged.drop(columns=['TeamID'], inplace=True)

    # For the loser rolling stats:
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                            on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    # Merge loser stats using the losing team key.
    df_merged = df_merged.merge(df_loser, left_on=['Season', 'DayNum', 'LTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)

    
    print("merged")

    # Drop games where either team does not have enough prior matches.
    required_cols = [col for col in df_merged.columns if col.endswith('_avg')]
    df_merged = df_merged.dropna(subset=required_cols)
    
    print("dropped NaNs")

    # Now, create training samples.
    X_rows = []
    y_rows = []
    
    # List of feature columns for winner and loser (order is important).
    w_features = [col for col in df_merged.columns if col.startswith('W_') and col.endswith('_avg')]
    l_features = [col for col in df_merged.columns if col.startswith('L_') and col.endswith('_avg')]
    
    # Use tqdm to show progress while iterating over the merged DataFrame.
    for _, row in tqdm(df_merged.iterrows(), total=len(df_merged), desc="Building training samples"):
        # Sample 1: team 1 is winner, team 2 is loser, label 1
        sample1 = pd.concat([row[w_features], row[l_features]])
        X_rows.append(sample1)
        y_rows.append(1)
        # Sample 2: team 1 is loser, team 2 is winner, label 0
        sample2 = pd.concat([row[l_features], row[w_features]])
        X_rows.append(sample2)
        y_rows.append(0)
    
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df




In [24]:
start_year = 2019
X, y = create_training_data_vectorized(df, start_year, n_matches=5, n_seasons=3)
print(X.head())
print(y.head())

copy created
long_df created
rolling stats computed
rolling cols selected
winner stats merged
loser stats merged
merged
dropped NaNs


Building training samples: 100%|██████████| 41993/41993 [00:38<00:00, 1105.04it/s]


     W_Score_avg  W_FGM_avg  W_FGA_avg  W_FGM3_avg  W_FGA3_avg  W_FTM_avg  \
580         86.6       30.8       62.0         9.6        28.8       15.4   
580         86.6       30.8       62.0         9.6        28.8       15.4   
691         79.2       27.4       61.6         4.6        15.8       19.8   
691         79.2       27.4       61.6         4.6        15.8       19.8   
730         89.8       30.4       62.0         9.2        22.6       19.8   

     W_FTA_avg  W_OR_avg  W_DR_avg  W_Ast_avg  ...  L_OR_avg  L_DR_avg  \
580       22.4       9.0      24.0       19.2  ...      11.0      24.4   
580       22.4       9.0      24.0       19.2  ...      11.0      24.4   
691       29.0      16.8      27.8       10.6  ...      13.4      28.6   
691       29.0      16.8      27.8       10.6  ...      13.4      28.6   
730       29.0      14.0      29.8       19.6  ...      15.8      28.2   

     L_Ast_avg  L_TO_avg  L_Stl_avg  L_Blk_avg  L_PF_avg  L_home_avg  \
580        9.6      

In [27]:
print(X.shape)
print(y.shape)

(83986, 34)
(83986,)


In [35]:
X.columns

Index(['W_Score_avg', 'W_FGM_avg', 'W_FGA_avg', 'W_FGM3_avg', 'W_FGA3_avg',
       'W_FTM_avg', 'W_FTA_avg', 'W_OR_avg', 'W_DR_avg', 'W_Ast_avg',
       'W_TO_avg', 'W_Stl_avg', 'W_Blk_avg', 'W_PF_avg', 'W_home_avg',
       'W_away_avg', 'W_neutral_avg', 'L_Score_avg', 'L_FGM_avg', 'L_FGA_avg',
       'L_FGM3_avg', 'L_FGA3_avg', 'L_FTM_avg', 'L_FTA_avg', 'L_OR_avg',
       'L_DR_avg', 'L_Ast_avg', 'L_TO_avg', 'L_Stl_avg', 'L_Blk_avg',
       'L_PF_avg', 'L_home_avg', 'L_away_avg', 'L_neutral_avg'],
      dtype='object')

In [28]:
data = X.copy()
data['target'] = y

In [29]:
data.to_csv("../../data/training/season_2019.csv", index=False)

# Attempt 2

In [53]:
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, brier_score_loss, classification_report

# -------------------------------
# 1. Load Regular Season Results
# -------------------------------
df_reg = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")
print("Regular season results loaded:", df_reg.shape)
print("Seasons in data:", sorted(df_reg['Season'].unique()))

# ----------------------------------------------
# 2. Split into Training (≤2019) and Test (2020)
# ----------------------------------------------
df_train = df_reg[df_reg['Season'] <= 2019].copy()
print("Training seasons:", sorted(df_train['Season'].unique()))
df_test  = df_reg[df_reg['Season'] == 2020].copy()
print("Test season:", sorted(df_test['Season'].unique()))

# -------------------------------------------------
# 3. Create Training and Test Samples Using Your Function
# -------------------------------------------------
# Note: start_year here is used as a cutoff (with n_seasons backtracking)
# For training, we set start_year=2019 so that games from (2019-n_seasons) onward are used.
X_train, y_train = create_training_data_vectorized(df_train, start_year=2019, n_matches=5, n_seasons=3)
print("Training data shapes: X_train =", X_train.shape, ", y_train =", y_train.shape)

# For testing, we use season 2020 (set start_year=2020) – note that if a team in 2020 has not played enough games, that game will be dropped.
X_test, y_test = create_training_data_vectorized(df_test, start_year=2020, n_matches=5, n_seasons=3)
print("Test data shapes: X_test =", X_test.shape, ", y_test =", y_test.shape)

# -------------------------------------------------
# 4. Train a Model on Training Data and Test on 2020 Data
# -------------------------------------------------
# We'll build a simple pipeline with scaling and logistic regression.
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000))
])
pipe.fit(X_train, y_train)

# Evaluate on test data.
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_prob)

print("\nTest Results:")
print("  Accuracy: {:.2f}%".format(acc * 100))
print("  Brier Score: {:.4f}".format(brier))
print("  Classification Report:")
print(classification_report(y_test, y_pred))


Regular season results loaded: (117748, 34)
Seasons in data: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Training seasons: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Test season: [np.int64(2020)]
copy created
long_df created
rolling stats computed
rolling cols selected
winner stats merged
loser stats merged
merged
dropped NaNs


Building training samples: 100%|██████████| 17737/17737 [00:15<00:00, 1110.06it/s]


Training data shapes: X_train = (35474, 34) , y_train = (35474,)
copy created
long_df created
rolling stats computed
rolling cols selected
winner stats merged
loser stats merged
merged
dropped NaNs


Building training samples: 100%|██████████| 4354/4354 [00:03<00:00, 1094.88it/s]


Test data shapes: X_test = (8708, 34) , y_test = (8708,)

Test Results:
  Accuracy: 50.00%
  Brier Score: 0.2500
  Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      4354
           1       0.00      0.00      0.00      4354

    accuracy                           0.50      8708
   macro avg       0.25      0.50      0.33      8708
weighted avg       0.25      0.50      0.33      8708



c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels w

# Not working

In [46]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, brier_score_loss

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("\nLogistic Regression Accuracy: {:.2f}%".format(acc_lr * 100))
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 48.48%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.56      0.52      8337
           1       0.49      0.42      0.45      8461

    accuracy                           0.48     16798
   macro avg       0.49      0.49      0.48     16798
weighted avg       0.49      0.48      0.48     16798



In [34]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("\nRandom Forest Accuracy: {:.2f}%".format(acc_rf * 100))
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))



Random Forest Accuracy: 9.83%
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.10      0.10      0.10      8337
           1       0.10      0.10      0.10      8461

    accuracy                           0.10     16798
   macro avg       0.10      0.10      0.10     16798
weighted avg       0.10      0.10      0.10     16798



In [36]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)
print("\nGradient Boosting Accuracy: {:.2f}%".format(acc_gb * 100))
print("Gradient Boosting Classification Report:")
print(classification_report(y_test, y_pred_gb))


Gradient Boosting Accuracy: 43.28%
Gradient Boosting Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.52      0.48      8337
           1       0.42      0.34      0.38      8461

    accuracy                           0.43     16798
   macro avg       0.43      0.43      0.43     16798
weighted avg       0.43      0.43      0.43     16798



In [40]:
# --- Define Models ---
# Logistic Regression (already calibrated)
lr = LogisticRegression(max_iter=1000)

# For Random Forest and Gradient Boosting we use calibration.
rf = CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42), cv=3, method='isotonic')
gb = CalibratedClassifierCV(GradientBoostingClassifier(n_estimators=100, random_state=42), cv=3, method='isotonic')

models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

# --- Train Models and Evaluate ---
for name, model in models.items():
    model.fit(X_train, y_train)
    # Predict probabilities (the probability for class 1)
    probs = model.predict_proba(X_test)[:, 1]
    # Predict labels for accuracy and classification report
    pred_labels = model.predict(X_test)
    acc = accuracy_score(y_test, pred_labels)
    brier = brier_score_loss(y_test, probs)
    
    print(f"\n{name}:")
    print("  Accuracy: {:.2f}%".format(acc * 100))
    print("  Brier Score: {:.4f}".format(brier))
    print("  Classification Report:")
    print(classification_report(y_test, pred_labels))


Logistic Regression:
  Accuracy: 48.48%
  Brier Score: 0.2503
  Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.56      0.52      8337
           1       0.49      0.42      0.45      8461

    accuracy                           0.48     16798
   macro avg       0.49      0.49      0.48     16798
weighted avg       0.49      0.48      0.48     16798


Random Forest:
  Accuracy: 49.63%
  Brier Score: 0.2500
  Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.66      8337
           1       0.00      0.00      0.00      8461

    accuracy                           0.50     16798
   macro avg       0.25      0.50      0.33     16798
weighted avg       0.25      0.50      0.33     16798



c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels w


Gradient Boosting:
  Accuracy: 49.63%
  Brier Score: 0.2500
  Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.66      8337
           1       0.00      0.00      0.00      8461

    accuracy                           0.50     16798
   macro avg       0.25      0.50      0.33     16798
weighted avg       0.25      0.50      0.33     16798



c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels w

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [47]:
##############################################
# 1. Optimized Logistic Regression Pipeline  #
##############################################

# Logistic Regression with scaling and balanced class weights.
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', 
     Pipeline([('lr_inner', 
                # Dummy estimator; we'll tune its parameters via GridSearchCV.
                # We'll override parameters in GridSearch.
                # Here we use LogisticRegression.
                __import__('sklearn.linear_model').linear_model.LogisticRegression(max_iter=1000, class_weight='balanced')
               )])
    )
])

# Instead of wrapping a nested pipeline, we can directly build one:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', 
     __import__('sklearn.linear_model').linear_model.LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid_lr = {
    'lr__C': [0.01, 0.1, 1, 10, 100]
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_lr.fit(X_train, y_train)
print("Best parameters for Logistic Regression:", grid_lr.best_params_)

lr_best = grid_lr.best_estimator_
y_pred_lr = lr_best.predict(X_test)
y_prob_lr = lr_best.predict_proba(X_test)[:, 1]
acc_lr = accuracy_score(y_test, y_pred_lr)
brier_lr = brier_score_loss(y_test, y_prob_lr)

print("\nOptimized Logistic Regression:")
print("  Accuracy: {:.2f}%".format(acc_lr * 100))
print("  Brier Score: {:.4f}".format(brier_lr))
print("  Classification Report:")
print(classification_report(y_test, y_pred_lr))

Best parameters for Logistic Regression: {'lr__C': 0.01}

Optimized Logistic Regression:
  Accuracy: 48.30%
  Brier Score: 0.2502
  Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.48      0.48      8337
           1       0.49      0.48      0.48      8461

    accuracy                           0.48     16798
   macro avg       0.48      0.48      0.48     16798
weighted avg       0.48      0.48      0.48     16798



In [48]:
from sklearn.ensemble import RandomForestClassifier

# For Random Forest, scaling is not necessary but class_weight can help.
pipe_rf = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])

param_grid_rf = {
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best parameters for Random Forest:", grid_rf.best_params_)

rf_best = grid_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test)
y_prob_rf = rf_best.predict_proba(X_test)[:, 1]
acc_rf = accuracy_score(y_test, y_pred_rf)
brier_rf = brier_score_loss(y_test, y_prob_rf)

print("\nOptimized Random Forest:")
print("  Accuracy: {:.2f}%".format(acc_rf * 100))
print("  Brier Score: {:.4f}".format(brier_rf))
print("  Classification Report:")
print(classification_report(y_test, y_pred_rf))

Best parameters for Random Forest: {'rf__max_depth': 10, 'rf__min_samples_split': 10}

Optimized Random Forest:
  Accuracy: 23.53%
  Brier Score: 0.2561
  Classification Report:
              precision    recall  f1-score   support

           0       0.22      0.21      0.22      8337
           1       0.25      0.26      0.25      8461

    accuracy                           0.24     16798
   macro avg       0.23      0.24      0.23     16798
weighted avg       0.23      0.24      0.23     16798



In [49]:
##############################################
# 3. Optimized Gradient Boosting Pipeline    #
##############################################

from sklearn.ensemble import GradientBoostingClassifier

pipe_gb = Pipeline([
    # For GB, scaling is not required.
    ('gb', GradientBoostingClassifier(random_state=42))
])

param_grid_gb = {
    'gb__n_estimators': [100, 200],
    'gb__learning_rate': [0.01, 0.1, 0.2],
    'gb__max_depth': [3, 5, 7]
}

grid_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_gb.fit(X_train, y_train)
print("Best parameters for Gradient Boosting:", grid_gb.best_params_)

gb_best = grid_gb.best_estimator_
y_pred_gb = gb_best.predict(X_test)
y_prob_gb = gb_best.predict_proba(X_test)[:, 1]
acc_gb = accuracy_score(y_test, y_pred_gb)
brier_gb = brier_score_loss(y_test, y_prob_gb)

print("\nOptimized Gradient Boosting:")
print("  Accuracy: {:.2f}%".format(acc_gb * 100))
print("  Brier Score: {:.4f}".format(brier_gb))
print("  Classification Report:")
print(classification_report(y_test, y_pred_gb))

Best parameters for Gradient Boosting: {'gb__learning_rate': 0.01, 'gb__max_depth': 3, 'gb__n_estimators': 100}

Optimized Gradient Boosting:
  Accuracy: 47.42%
  Brier Score: 0.2504
  Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.79      0.60      8337
           1       0.44      0.17      0.24      8461

    accuracy                           0.47     16798
   macro avg       0.46      0.48      0.42     16798
weighted avg       0.46      0.47      0.42     16798



In [51]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Load the CSV file with the training data.
data_path = "../../data/training/season_2019.csv"
df = pd.read_csv(data_path)
print("Data loaded from", data_path)
print("Data shape:", df.shape)

# Assume the CSV has columns like:
# W_Score_avg, W_FGM_avg, W_FGA_avg, ..., L_Score_avg, L_FGM_avg, L_FGA_avg, ..., and 'target'
# Let's create differential features for every corresponding statistic.

# Get list of columns for team 1 (winner stats) and team 2 (loser stats)
w_columns = [col for col in df.columns if col.startswith("W_") and col.endswith("_avg")]
l_columns = [col for col in df.columns if col.startswith("L_") and col.endswith("_avg")]

# Sort columns to make sure corresponding features align.
w_columns.sort()
l_columns.sort()

# Create a new DataFrame for the differential features.
diff_features = pd.DataFrame()
for w_col, l_col in zip(w_columns, l_columns):
    # Create a new column name like "diff_Score_avg" from "W_Score_avg"
    diff_name = w_col.replace("W_", "diff_")
    diff_features[diff_name] = df[w_col] - df[l_col]

# The target remains the same (1 if team1 wins, 0 otherwise)
y = df['target']

# Optional: Check the new feature names.
print("Differential feature columns:", diff_features.columns.tolist())

# Split the data.
X_train, X_test, y_train, y_test = train_test_split(diff_features, y, test_size=0.2, random_state=42)
print("Training samples:", X_train.shape, "Test samples:", X_test.shape)

# Build a pipeline: scaling and logistic regression.
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Tune the regularization strength to minimize the Brier score.
param_grid = {
    'lr__C': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid.fit(X_train, y_train)
print("Best parameters for differential Logistic Regression:", grid.best_params_)

# Evaluate on the test set.
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_prob)

print("\nDifferential Logistic Regression Results:")
print("  Accuracy: {:.2f}%".format(acc * 100))
print("  Brier Score: {:.4f}".format(brier))
print("  Classification Report:")
print(classification_report(y_test, y_pred))


Data loaded from ../../data/training/season_2019.csv
Data shape: (83986, 35)
Differential feature columns: ['diff_Ast_avg', 'diff_Blk_avg', 'diff_DR_avg', 'diff_FGA3_avg', 'diff_FGA_avg', 'diff_FGM3_avg', 'diff_FGM_avg', 'diff_FTA_avg', 'diff_FTM_avg', 'diff_OR_avg', 'diff_PF_avg', 'diff_Score_avg', 'diff_Stl_avg', 'diff_TO_avg', 'diff_away_avg', 'diff_home_avg', 'diff_neutral_avg']
Training samples: (67188, 17) Test samples: (16798, 17)
Best parameters for differential Logistic Regression: {'lr__C': 0.01}

Differential Logistic Regression Results:
  Accuracy: 50.56%
  Brier Score: 0.2500
  Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50      8369
           1       0.51      0.51      0.51      8429

    accuracy                           0.51     16798
   macro avg       0.51      0.51      0.51     16798
weighted avg       0.51      0.51      0.51     16798



In [52]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV

# -----------------------------
# 1. Load and Prepare the Data
# -----------------------------
data_path = "../../data/training/season_2019.csv"
df = pd.read_csv(data_path)
print("Data loaded from", data_path)
print("Data shape:", df.shape)

# Create differential features: subtract each L_..._avg from its corresponding W_..._avg
w_columns = [col for col in df.columns if col.startswith("W_") and col.endswith("_avg")]
l_columns = [col for col in df.columns if col.startswith("L_") and col.endswith("_avg")]
w_columns.sort()
l_columns.sort()

diff_features = pd.DataFrame()
for w_col, l_col in zip(w_columns, l_columns):
    diff_name = w_col.replace("W_", "diff_")
    diff_features[diff_name] = df[w_col] - df[l_col]

# The target column remains unchanged.
y = df['target']

print("Differential feature columns:", diff_features.columns.tolist())
print("Training samples:", diff_features.shape)

# Split the data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(diff_features, y, test_size=0.2, random_state=42)
print("Train/Test split:", X_train.shape, X_test.shape)

# -------------------------------------------------------
# 2. Model 1: Logistic Regression (Extended Regularization)
# -------------------------------------------------------
# We extend the grid to allow less regularization (larger C) so that the model isn’t forced to predict near 0.5.
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
param_grid_lr = {
    'lr__C': [0.01, 0.1, 1, 10, 100, 1000]
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
print("Best parameters for Logistic Regression:", grid_lr.best_params_)

y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:,1]
acc_lr = accuracy_score(y_test, y_pred_lr)
brier_lr = brier_score_loss(y_test, y_prob_lr)
print("\nLogistic Regression Results:")
print("  Accuracy: {:.2f}%".format(acc_lr * 100))
print("  Brier Score: {:.4f}".format(brier_lr))
print("  Classification Report:")
print(classification_report(y_test, y_pred_lr))


# -------------------------------------------------------
# 3. Model 2: Gradient Boosting Classifier (Tuned)
# -------------------------------------------------------
pipe_gb = Pipeline([
    ('gb', GradientBoostingClassifier(random_state=42))
])
param_grid_gb = {
    'gb__n_estimators': [100, 200],
    'gb__learning_rate': [0.01, 0.1, 0.2],
    'gb__max_depth': [3, 5, 7]
}
grid_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_gb.fit(X_train, y_train)
best_gb = grid_gb.best_estimator_
print("Best parameters for Gradient Boosting:", grid_gb.best_params_)

y_pred_gb = best_gb.predict(X_test)
y_prob_gb = best_gb.predict_proba(X_test)[:,1]
acc_gb = accuracy_score(y_test, y_pred_gb)
brier_gb = brier_score_loss(y_test, y_prob_gb)
print("\nGradient Boosting Results:")
print("  Accuracy: {:.2f}%".format(acc_gb * 100))
print("  Brier Score: {:.4f}".format(brier_gb))
print("  Classification Report:")
print(classification_report(y_test, y_pred_gb))


# -------------------------------------------------------
# 4. Model 3: Random Forest Classifier (Tuned)
# -------------------------------------------------------
pipe_rf = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])
param_grid_rf = {
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5, 10]
}
grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print("Best parameters for Random Forest:", grid_rf.best_params_)

y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:,1]
acc_rf = accuracy_score(y_test, y_pred_rf)
brier_rf = brier_score_loss(y_test, y_prob_rf)
print("\nRandom Forest Results:")
print("  Accuracy: {:.2f}%".format(acc_rf * 100))
print("  Brier Score: {:.4f}".format(brier_rf))
print("  Classification Report:")
print(classification_report(y_test, y_pred_rf))


# -------------------------------------------------------
# 5. Model 4: MLP (Neural Network) Classifier (Tuned)
# -------------------------------------------------------
pipe_nn = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=1000, random_state=42))
])
param_grid_nn = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__alpha': [0.0001, 0.001, 0.01]
}
grid_nn = GridSearchCV(pipe_nn, param_grid_nn, cv=5, scoring='neg_brier_score', n_jobs=-1)
grid_nn.fit(X_train, y_train)
best_nn = grid_nn.best_estimator_
print("Best parameters for MLP:", grid_nn.best_params_)

y_pred_nn = best_nn.predict(X_test)
y_prob_nn = best_nn.predict_proba(X_test)[:,1]
acc_nn = accuracy_score(y_test, y_pred_nn)
brier_nn = brier_score_loss(y_test, y_prob_nn)
print("\nMLP Classifier Results:")
print("  Accuracy: {:.2f}%".format(acc_nn * 100))
print("  Brier Score: {:.4f}".format(brier_nn))
print("  Classification Report:")
print(classification_report(y_test, y_pred_nn))


# -------------------------------------------------------
# 6. Option: Further Calibration
# -------------------------------------------------------
# If the best models still predict nearly constant probabilities (Brier ~0.2500),
# you can try additional calibration. For example, wrapping the best logistic regression:
calibrated_lr = CalibratedClassifierCV(best_lr, cv=5, method='isotonic')
calibrated_lr.fit(X_train, y_train)
y_prob_cal = calibrated_lr.predict_proba(X_test)[:,1]
brier_cal = brier_score_loss(y_test, y_prob_cal)
print("\nCalibrated Logistic Regression Brier Score: {:.4f}".format(brier_cal))


Data loaded from ../../data/training/season_2019.csv
Data shape: (83986, 35)
Differential feature columns: ['diff_Ast_avg', 'diff_Blk_avg', 'diff_DR_avg', 'diff_FGA3_avg', 'diff_FGA_avg', 'diff_FGM3_avg', 'diff_FGM_avg', 'diff_FTA_avg', 'diff_FTM_avg', 'diff_OR_avg', 'diff_PF_avg', 'diff_Score_avg', 'diff_Stl_avg', 'diff_TO_avg', 'diff_away_avg', 'diff_home_avg', 'diff_neutral_avg']
Training samples: (83986, 17)
Train/Test split: (67188, 17) (16798, 17)
Best parameters for Logistic Regression: {'lr__C': 0.01}

Logistic Regression Results:
  Accuracy: 50.56%
  Brier Score: 0.2500
  Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50      8369
           1       0.51      0.51      0.51      8429

    accuracy                           0.51     16798
   macro avg       0.51      0.51      0.51     16798
weighted avg       0.51      0.51      0.51     16798



KeyboardInterrupt: 

# Adding team ELO

In [56]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def add_elo_ratings(df, base_rating=1500, K=20):
    """
    Compute and merge Elo ratings into the game-level DataFrame.
    
    For each game (sorted by Season and DayNum), retrieves the pre-game Elo ratings
    for the winning and losing teams (default to base_rating if not found), computes 
    the expected outcome, updates the ratings, and adds two new columns:
      - 'W_Elo_pre': pre-game Elo for the winning team.
      - 'L_Elo_pre': pre-game Elo for the losing team.
    
    Parameters:
      df (pd.DataFrame): Must include 'Season', 'DayNum', 'WTeamID', and 'LTeamID'.
      base_rating (int): Starting Elo rating for each team.
      K (int): K–factor for Elo updates.
    
    Returns:
      pd.DataFrame: Original DataFrame with added columns 'W_Elo_pre' and 'L_Elo_pre'.
    """
    # Sort by season and day to process in chronological order.
    df = df.sort_values(by=['Season', 'DayNum']).copy()
    
    team_ratings = {}  # dictionary to hold current Elo ratings
    elo_pre = []       # list to store pre-game ratings for each game
    
    # Process each game sequentially.
    for idx, row in df.iterrows():
        winner = row['WTeamID']
        loser = row['LTeamID']
        
        rating_w = team_ratings.get(winner, base_rating)
        rating_l = team_ratings.get(loser, base_rating)
        
        # Save pre-game Elo ratings.
        elo_pre.append((rating_w, rating_l))
        
        # Expected outcomes.
        expected_w = 1 / (1 + 10 ** ((rating_l - rating_w) / 400))
        expected_l = 1 - expected_w
        
        # Update ratings.
        team_ratings[winner] = rating_w + K * (1 - expected_w)
        team_ratings[loser] = rating_l + K * (0 - expected_l)
    
    # Add new columns.
    df['W_Elo_pre'] = [x[0] for x in elo_pre]
    df['L_Elo_pre'] = [x[1] for x in elo_pre]
    
    return df

def create_long_df(df):
    """
    Convert the game-level DataFrame into a long format where each row corresponds 
    to a team’s performance in a game.
    """
    # Process winner rows.
    winners = df.copy()
    winners['TeamID'] = winners['WTeamID']
    winners['Score'] = winners['WScore']
    winners['FGM']   = winners['WFGM']
    winners['FGA']   = winners['WFGA']
    winners['FGM3']  = winners['WFGM3']
    winners['FGA3']  = winners['WFGA3']
    winners['FTM']   = winners['WFTM']
    winners['FTA']   = winners['WFTA']
    winners['OR']    = winners['WOR']
    winners['DR']    = winners['WDR']
    winners['Ast']   = winners['WAst']
    winners['TO']    = winners['WTO']
    winners['Stl']   = winners['WStl']
    winners['Blk']   = winners['WBlk']
    winners['PF']    = winners['WPF']
    winners['Loc']   = winners['WLoc']
    winners['is_winner'] = 1

    # Process loser rows.
    losers = df.copy()
    losers['TeamID'] = losers['LTeamID']
    losers['Score'] = losers['LScore']
    losers['FGM']   = losers['LFGM']
    losers['FGA']   = losers['LFGA']
    losers['FGM3']  = losers['LFGM3']
    losers['FGA3']  = losers['LFGA3']
    losers['FTM']   = losers['LFTM']
    losers['FTA']   = losers['LFTA']
    losers['OR']    = losers['LOR']
    losers['DR']    = losers['LDR']
    losers['Ast']   = losers['LAst']
    losers['TO']    = losers['LTO']
    losers['Stl']   = losers['LStl']
    losers['Blk']   = losers['LBlk']
    losers['PF']    = losers['LPF']
    losers['Loc'] = losers['WLoc'].map(lambda x: 'A' if x=='H' else ('H' if x=='A' else 'N'))
    losers['is_winner'] = 0

    common_cols = ['Season', 'DayNum', 'TeamID', 'Score', 'FGM', 'FGA', 
                   'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 
                   'Stl', 'Blk', 'PF', 'Loc', 'is_winner']
    
    long_df = pd.concat([winners[common_cols], losers[common_cols]], ignore_index=True)
    return long_df

def compute_rolling_stats(long_df, n_matches=5):
    """
    Compute the rolling (previous n_matches) average for each statistic for each team in a season.
    A shift of 1 is applied so that the current game is not included.
    Also computes the rolling averages for one-hot encoded location indicators.
    """
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
                 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    long_df['home']    = (long_df['Loc'] == 'H').astype(int)
    long_df['away']    = (long_df['Loc'] == 'A').astype(int)
    long_df['neutral'] = (long_df['Loc'] == 'N').astype(int)
    onehot_cols = ['home', 'away', 'neutral']
    
    long_df = long_df.sort_values(by=['TeamID', 'Season', 'DayNum'])
    
    for col in stat_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    for col in onehot_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    return long_df

def create_training_data_vectorized_with_elo(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (X and y) using vectorized operations.
    
    The function processes games from seasons >= (start_year - n_seasons), computes rolling averages,
    and also computes Elo ratings (with a base of 1500). For each game, it then creates two training
    samples:
      - [winner_rolling features | loser_rolling features | winner_Elo and loser_Elo] with label 1 (team 1 wins)
      - [loser_rolling features | winner_rolling features | loser_Elo and winner_Elo] with label 0 (team 1 loses)
    
    Returns:
      X_df (pd.DataFrame): Each row is the concatenated features for team1 and team2.
      y_df (pd.Series): Corresponding target label (1 if team1 wins, 0 otherwise).
    """
    # Filter games for the desired seasons.
    df_filtered = df[df['Season'] >= start_year - n_seasons].copy()
    print("copy created")
    
    # --- Add Elo ratings to the game-level DataFrame ---
    df_filtered = add_elo_ratings(df_filtered, base_rating=1500, K=20)
    print("Elo ratings computed")
    
    # Create long-format DataFrame.
    long_df = create_long_df(df_filtered)
    print("long_df created")
    
    # Compute rolling averages.
    long_df = compute_rolling_stats(long_df, n_matches=n_matches)
    print("rolling stats computed")
    
    # Select rolling average columns.
    rolling_cols = [col for col in long_df.columns if col.endswith('_avg')]
    print("rolling cols selected")
    
    # --- Merge rolling stats back into game-level DataFrame ---
    # Merge winner stats.
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    print("winner stats merged")
    
    # Merge loser stats.
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    print("loser stats merged")
    
    # Merge winner rolling stats using proper keys.
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    df_merged = df_filtered.merge(df_winner, left_on=['Season', 'DayNum', 'WTeamID'],
                                  right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)
    
    # Merge loser rolling stats.
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    df_merged = df_merged.merge(df_loser, left_on=['Season', 'DayNum', 'LTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)
    print("merged")
    
    # Drop games with insufficient prior matches.
    required_cols = [col for col in df_merged.columns if col.endswith('_avg')]
    df_merged = df_merged.dropna(subset=required_cols)
    print("dropped NaNs")
    
    # --- Build training samples ---
    X_rows = []
    y_rows = []
    
    # Select feature columns from the merged DataFrame.
    w_features = [col for col in df_merged.columns if col.startswith('W_') and col.endswith('_avg')]
    l_features = [col for col in df_merged.columns if col.startswith('L_') and col.endswith('_avg')]
    # Also include the Elo ratings from df_filtered (which remain in df_merged)
    if 'W_Elo_pre' in df_merged.columns and 'L_Elo_pre' in df_merged.columns:
        w_features.append('W_Elo_pre')
        l_features.append('L_Elo_pre')
    
    # Create two samples per game: one as [winner_features | loser_features] and one flipped.
    for _, row in tqdm(df_merged.iterrows(), total=len(df_merged), desc="Building training samples"):
        sample1 = pd.concat([row[w_features], row[l_features]])
        X_rows.append(sample1)
        y_rows.append(1)
        
        sample2 = pd.concat([row[l_features], row[w_features]])
        X_rows.append(sample2)
        y_rows.append(0)
    
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def add_elo_ratings(df, base_rating=1500, K=20):
    """
    Compute and merge Elo ratings into the game-level DataFrame.
    
    For each game (sorted by Season and DayNum), retrieves the pre-game Elo ratings
    for the winning and losing teams (default to base_rating if not found), computes 
    the expected outcome, updates the ratings, and adds two new columns:
      - 'W_Elo_pre': pre-game Elo for the winning team.
      - 'L_Elo_pre': pre-game Elo for the losing team.
    """
    df = df.sort_values(by=['Season', 'DayNum']).copy()
    team_ratings = {}
    elo_pre = []
    
    for idx, row in df.iterrows():
        winner = row['WTeamID']
        loser = row['LTeamID']
        
        rating_w = team_ratings.get(winner, base_rating)
        rating_l = team_ratings.get(loser, base_rating)
        
        # Save pre-game Elo ratings.
        elo_pre.append((rating_w, rating_l))
        
        expected_w = 1 / (1 + 10 ** ((rating_l - rating_w) / 400))
        expected_l = 1 - expected_w
        
        team_ratings[winner] = rating_w + K * (1 - expected_w)
        team_ratings[loser] = rating_l + K * (0 - expected_l)
    
    df['W_Elo_pre'] = [x[0] for x in elo_pre]
    df['L_Elo_pre'] = [x[1] for x in elo_pre]
    return df

def create_long_df(df):
    """
    Convert the game-level DataFrame into a long format where each row corresponds 
    to a team’s performance in a game.
    """
    # Process winner rows.
    winners = df.copy()
    winners['TeamID'] = winners['WTeamID']
    winners['Score'] = winners['WScore']
    winners['FGM']   = winners['WFGM']
    winners['FGA']   = winners['WFGA']
    winners['FGM3']  = winners['WFGM3']
    winners['FGA3']  = winners['WFGA3']
    winners['FTM']   = winners['WFTM']
    winners['FTA']   = winners['WFTA']
    winners['OR']    = winners['WOR']
    winners['DR']    = winners['WDR']
    winners['Ast']   = winners['WAst']
    winners['TO']    = winners['WTO']
    winners['Stl']   = winners['WStl']
    winners['Blk']   = winners['WBlk']
    winners['PF']    = winners['WPF']
    winners['Loc']   = winners['WLoc']
    winners['is_winner'] = 1

    # Process loser rows.
    losers = df.copy()
    losers['TeamID'] = losers['LTeamID']
    losers['Score'] = losers['LScore']
    losers['FGM']   = losers['LFGM']
    losers['FGA']   = losers['LFGA']
    losers['FGM3']  = losers['LFGM3']
    losers['FGA3']  = losers['LFGA3']
    losers['FTM']   = losers['LFTM']
    losers['FTA']   = losers['LFTA']
    losers['OR']    = losers['LOR']
    losers['DR']    = losers['LDR']
    losers['Ast']   = losers['LAst']
    losers['TO']    = losers['LTO']
    losers['Stl']   = losers['LStl']
    losers['Blk']   = losers['LBlk']
    losers['PF']    = losers['LPF']
    losers['Loc'] = losers['WLoc'].map(lambda x: 'A' if x=='H' else ('H' if x=='A' else 'N'))
    losers['is_winner'] = 0

    common_cols = ['Season', 'DayNum', 'TeamID', 'Score', 'FGM', 'FGA', 
                   'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 
                   'Stl', 'Blk', 'PF', 'Loc', 'is_winner']
    
    long_df = pd.concat([winners[common_cols], losers[common_cols]], ignore_index=True)
    return long_df

def compute_rolling_stats(long_df, n_matches=5):
    """
    Compute the rolling (previous n_matches) average for each statistic for each team in a season.
    A shift of 1 is applied so that the current game is not included.
    Also computes the rolling averages for one-hot encoded location indicators.
    
    To reduce NaNs, we use min_periods=1 so that even if fewer than n_matches are available,
    a rolling average is computed.
    """
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
                 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    long_df['home']    = (long_df['Loc'] == 'H').astype(int)
    long_df['away']    = (long_df['Loc'] == 'A').astype(int)
    long_df['neutral'] = (long_df['Loc'] == 'N').astype(int)
    onehot_cols = ['home', 'away', 'neutral']
    
    long_df = long_df.sort_values(by=['TeamID', 'Season', 'DayNum'])
    
    # Use min_periods=1 to compute an average from available games.
    for col in stat_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col]\
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    for col in onehot_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col]\
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    return long_df

def create_training_data_vectorized_with_elo(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (X and y) using vectorized operations.
    
    This function processes games from seasons >= (start_year - n_seasons), computes rolling averages,
    and computes Elo ratings (with a base of 1500). For each game, it creates two training samples:
      - One sample where team1's features (including Elo) come from the winning team and team2's from the loser, with label 1.
      - A second sample where the roles are flipped, with label 0.
    
    Season and DayNum are kept in the output for later use.
    
    Returns:
      X_df (pd.DataFrame): Each row is a dictionary of features for team1 and team2 (including Season and DayNum).
      y_df (pd.Series): Corresponding target label (1 if team1 wins, 0 otherwise).
    """
    # Filter games for the desired seasons.
    df_filtered = df[df['Season'] >= start_year - n_seasons].copy()
    print("copy created")
    
    # Add Elo ratings.
    df_filtered = add_elo_ratings(df_filtered, base_rating=1500, K=20)
    print("Elo ratings computed")
    
    # Create long-format DataFrame.
    long_df = create_long_df(df_filtered)
    print("long_df created")
    
    # Compute rolling averages.
    long_df = compute_rolling_stats(long_df, n_matches=n_matches)
    print("rolling stats computed")
    
    # Select rolling average columns.
    rolling_cols = [col for col in long_df.columns if col.endswith('_avg')]
    print("rolling cols selected")
    
    # --- Merge rolling stats back into game-level DataFrame ---
    # Winner stats.
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    print("winner stats merged")
    
    # Loser stats.
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    print("loser stats merged")
    
    # Merge winner stats into main DataFrame.
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    df_merged = df_filtered.merge(df_winner, left_on=['Season', 'DayNum', 'WTeamID'],
                                  right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)
    
    # Merge loser stats into main DataFrame.
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    df_merged = df_merged.merge(df_loser, left_on=['Season', 'DayNum', 'LTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)
    print("merged")
    
    # Drop games with insufficient data (if any remain).
    required_cols = [col for col in df_merged.columns if col.endswith('_avg')]
    df_merged = df_merged.dropna(subset=required_cols)
    print("dropped NaNs")
    
    # --- Build training samples ---
    X_rows = []
    y_rows = []
    
    # Select feature columns for rolling stats.
    w_features = [col for col in df_merged.columns if col.startswith('W_') and col.endswith('_avg')]
    l_features = [col for col in df_merged.columns if col.startswith('L_') and col.endswith('_avg')]
    # Also include the Elo ratings.
    if 'W_Elo_pre' in df_merged.columns and 'L_Elo_pre' in df_merged.columns:
        w_features.append('W_Elo_pre')
        l_features.append('L_Elo_pre')
    
    # Build samples: create two rows per game with flipped team order.
    for _, row in tqdm(df_merged.iterrows(), total=len(df_merged), desc="Building training samples"):
        # Get game-level info.
        season = row["Season"]
        day = row["DayNum"]
        
        # Sample 1: team1 = winner stats, team2 = loser stats.
        sample1 = {"Season": season, "DayNum": day}
        for feat in w_features:
            sample1[f"team1_{feat}"] = row[feat]
        for feat in l_features:
            sample1[f"team2_{feat}"] = row[feat]
        X_rows.append(sample1)
        y_rows.append(1)
        
        # Sample 2: flipped: team1 = loser stats, team2 = winner stats.
        sample2 = {"Season": season, "DayNum": day}
        for feat in l_features:
            sample2[f"team1_{feat}"] = row[feat]
        for feat in w_features:
            sample2[f"team2_{feat}"] = row[feat]
        X_rows.append(sample2)
        y_rows.append(0)
    
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df


In [5]:
df = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")

In [6]:
start_year = 2019
X, y = create_training_data_vectorized_with_elo(df, start_year, n_matches=5, n_seasons=3)
print(X.head())
print(y.head())

copy created
Elo ratings computed
long_df created
rolling stats computed
rolling cols selected
winner stats merged
loser stats merged
merged
dropped NaNs


Building training samples: 100%|██████████| 49707/49707 [00:07<00:00, 6419.90it/s]


   Season  DayNum  team1_W_Score_avg  team1_W_FGM_avg  team1_W_FGA_avg  \
0    2016      12               71.0             26.0             55.0   
1    2016      12                NaN              NaN              NaN   
2    2016      12               58.0             22.0             42.0   
3    2016      12                NaN              NaN              NaN   
4    2016      12               83.0             24.0             54.0   

   team1_W_FGM3_avg  team1_W_FGA3_avg  team1_W_FTM_avg  team1_W_FTA_avg  \
0               4.0              12.0             15.0             31.0   
1               NaN               NaN              NaN              NaN   
2               7.0              17.0              7.0             10.0   
3               NaN               NaN              NaN              NaN   
4              11.0              24.0             24.0             34.0   

   team1_W_OR_avg  ...  team2_W_DR_avg  team2_W_Ast_avg  team2_W_TO_avg  \
0            13.0  ...       

In [7]:
X.columns

Index(['Season', 'DayNum', 'team1_W_Score_avg', 'team1_W_FGM_avg',
       'team1_W_FGA_avg', 'team1_W_FGM3_avg', 'team1_W_FGA3_avg',
       'team1_W_FTM_avg', 'team1_W_FTA_avg', 'team1_W_OR_avg',
       'team1_W_DR_avg', 'team1_W_Ast_avg', 'team1_W_TO_avg',
       'team1_W_Stl_avg', 'team1_W_Blk_avg', 'team1_W_PF_avg',
       'team1_W_home_avg', 'team1_W_away_avg', 'team1_W_neutral_avg',
       'team1_W_Elo_pre', 'team2_L_Score_avg', 'team2_L_FGM_avg',
       'team2_L_FGA_avg', 'team2_L_FGM3_avg', 'team2_L_FGA3_avg',
       'team2_L_FTM_avg', 'team2_L_FTA_avg', 'team2_L_OR_avg',
       'team2_L_DR_avg', 'team2_L_Ast_avg', 'team2_L_TO_avg',
       'team2_L_Stl_avg', 'team2_L_Blk_avg', 'team2_L_PF_avg',
       'team2_L_home_avg', 'team2_L_away_avg', 'team2_L_neutral_avg',
       'team2_L_Elo_pre', 'team1_L_Score_avg', 'team1_L_FGM_avg',
       'team1_L_FGA_avg', 'team1_L_FGM3_avg', 'team1_L_FGA3_avg',
       'team1_L_FTM_avg', 'team1_L_FTA_avg', 'team1_L_OR_avg',
       'team1_L_DR_avg',

In [85]:
X['Season']

0        2016
1        2016
2        2016
3        2016
4        2016
         ... 
99409    2025
99410    2025
99411    2025
99412    2025
99413    2025
Name: Season, Length: 99414, dtype: int64

In [1]:
X[X['Season'] == 2025]

NameError: name 'X' is not defined

In [87]:
y.where(X['Season'] == 2025)

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
99409    0.0
99410    1.0
99411    0.0
99412    1.0
99413    0.0
Name: target, Length: 99414, dtype: float64

In [88]:
nan_proportion = X.isna().mean()
print(nan_proportion)

Season                 0.0
DayNum                 0.0
team1_W_Score_avg      0.5
team1_W_FGM_avg        0.5
team1_W_FGA_avg        0.5
                      ... 
team2_W_PF_avg         0.5
team2_W_home_avg       0.5
team2_W_away_avg       0.5
team2_W_neutral_avg    0.5
team2_W_Elo_pre        0.5
Length: 74, dtype: float64


# Wlasny kod

In [12]:
df = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")

In [13]:
import pandas as pd
from tqdm import tqdm


In [14]:
df.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='object')

In [15]:
games = df[['Season', 'DayNum', 'WTeamID', 'LTeamID']].drop_duplicates()

In [99]:
games

,Season,DayNum,WTeamID,LTeamID
0,2003,10,1104,1328
1,2003,10,1272,1393
2,2003,11,1266,1437
3,2003,11,1296,1457
4,2003,11,1400,1208
...,...,...,...,...
117743,2025,106,1461,1102
117744,2025,106,1462,1139
117745,2025,106,1466,1480
117746,2025,106,1468,1122


In [45]:
def compute_rolling_stats(match_df, Team_id, Season, DayNum, n_matches=5):
    stat_columns = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
        'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    # Filter matches where the team was involved
    match_df = match_df[(match_df['WTeamID'] == Team_id) | (match_df['LTeamID'] == Team_id)]
    
    # Sort by Season and DayNum
    match_df = match_df.sort_values(['Season', 'DayNum'])
    
    # Filter only matches before the given Season and DayNum
    match_df = match_df[(match_df['Season'] < Season) | ((match_df['Season'] == Season) & (match_df['DayNum'] < DayNum))]
    
    # Take the last `n_matches`
    last_n_matches = match_df.tail(n_matches).copy()  # Copy to avoid modifying the original dataframe

    for col in stat_columns:
        last_n_matches[col] = last_n_matches.apply(
            lambda row: row[f'W{col}'] if row['WTeamID'] == Team_id else row[f'L{col}'], axis=1
        )

    # Compute rolling averages for the selected matches
    rolling_stats = last_n_matches[stat_columns].mean()

    return last_n_matches, rolling_stats

# Example call
last_matches, rolling_stats = compute_rolling_stats(df, 1104, 2020, 0)
print(last_matches)
print(rolling_stats)


       Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  \
86905    2019     117     1261      74     1104      69    A      0    28   
87019    2019     120     1120      66     1104      60    A      0    24   
87176    2019     124     1116      82     1104      70    H      0    29   
87366    2019     129     1104      62     1279      57    N      0    24   
87444    2019     130     1246      73     1104      55    N      0    28   

       WFGA  ...  FGA3  FTM  FTA  OR  DR  Ast  TO  Stl  Blk  PF  
86905    66  ...    26   13   23  12  28   11  13    1    3  20  
87019    60  ...    19   11   16   8  26   10  19    7    5  16  
87176    65  ...    24   10   20  12  24    9  15    4    3  14  
87366    59  ...    17    9   18  17  33   12  16    7    3   9  
87444    59  ...    18   15   19  11  21    8  11    7    3  15  

[5 rows x 48 columns]
Score    63.2
FGM      22.4
FGA      57.0
FGM3      6.8
FGA3     20.8
FTM      11.6
FTA      19.2
OR       12.0
DR    

multiple cores version

In [21]:
pip install modin

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 5.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [36]:
import modin.pandas as pd  # Uses all available cores

def compute_rolling_stats_cpu(match_df, Team_id, Season, DayNum, n_matches=5):
    stat_columns = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
        'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    # Filter matches where the team was involved
    match_df = match_df[(match_df['WTeamID'] == Team_id) | (match_df['LTeamID'] == Team_id)]
    
    # Sort by Season and DayNum
    match_df = match_df.sort_values(['Season', 'DayNum'])
    
    # Filter only matches before the given Season and DayNum
    match_df = match_df[(match_df['Season'] < Season) | ((match_df['Season'] == Season) & (match_df['DayNum'] < DayNum))]
    
    # Take the last `n_matches`
    last_n_matches = match_df.tail(n_matches).copy()

    # Use vectorized assignment instead of apply
    for col in stat_columns:
        last_n_matches[col] = last_n_matches[f'W{col}'].where(
            last_n_matches['WTeamID'] == Team_id,
            other=last_n_matches[f'L{col}']
        )

    # Compute rolling averages for the selected matches
    rolling_stats = last_n_matches[stat_columns].mean()

    return last_n_matches, rolling_stats

# Example call (assuming `df` is now a Modin DataFrame or you convert it from pandas)
last_matches, rolling_stats = compute_rolling_stats_cpu(df, 1104, 2020, 0)
print(last_matches)
print(rolling_stats)


       Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  \
86905    2019     117     1261      74     1104      69    A      0    28   
87019    2019     120     1120      66     1104      60    A      0    24   
87176    2019     124     1116      82     1104      70    H      0    29   
87366    2019     129     1104      62     1279      57    N      0    24   
87444    2019     130     1246      73     1104      55    N      0    28   

       WFGA  ...  FGA3  FTM  FTA  OR  DR  Ast  TO  Stl  Blk  PF  
86905    66  ...    26   13   23  12  28   11  13    1    3  20  
87019    60  ...    19   11   16   8  26   10  19    7    5  16  
87176    65  ...    24   10   20  12  24    9  15    4    3  14  
87366    59  ...    17    9   18  17  33   12  16    7    3   9  
87444    59  ...    18   15   19  11  21    8  11    7    3  15  

[5 rows x 48 columns]
Score    63.2
FGM      22.4
FGA      57.0
FGM3      6.8
FGA3     20.8
FTM      11.6
FTA      19.2
OR       12.0
DR    

In [25]:
rolling_stats

Score    63.2
FGM      22.4
FGA      57.0
FGM3      6.8
FGA3     20.8
FTM      11.6
FTA      19.2
OR       12.0
DR       26.4
Ast      10.0
TO       14.8
Stl       5.2
Blk       3.4
PF       14.8
dtype: float64

In [47]:
rolling_stats_results = []
for _, row in tqdm(games.iterrows(), total=len(games), desc="Processing Games"):
    season, daynum, wteam, lteam = row['Season'], row['DayNum'], row['WTeamID'], row['LTeamID']
    
    wteam_stats = compute_rolling_stats(df, wteam, season, daynum)
    lteam_stats = compute_rolling_stats(df, lteam, season, daynum)

    rolling_stats_results.append({'Season': season, 'DayNum': daynum, 'TeamID': wteam, 'RollingStats': wteam_stats})
    rolling_stats_results.append({'Season': season, 'DayNum': daynum, 'TeamID': lteam, 'RollingStats': lteam_stats})

rolling_stats_df = pd.DataFrame(rolling_stats_results)
tools.display_dataframe_to_user(name="Rolling Stats Data", dataframe=rolling_stats_df)

Processing Games:   0%|          | 0/117748 [00:00<?, ?it/s]

TypeError: compute_rolling_stats() takes from 1 to 2 positional arguments but 4 were given

# Multicore

In [34]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import modin.pandas as pd  # Make sure you're using Modin for multi-core support

# Define a helper function to process a single game row.
def process_game(row):
    # Unpack the required game info
    season, daynum, wteam, lteam = row['Season'], row['DayNum'], row['WTeamID'], row['LTeamID']
    
    # Compute rolling stats for both teams.
    # Note: Ensure compute_rolling_stats_cpu is accessible here and uses Modin/pandas.
    wteam_stats = compute_rolling_stats_cpu(df, wteam, season, daynum)
    lteam_stats = compute_rolling_stats_cpu(df, lteam, season, daynum)
    
    # Return a list of dictionaries for each team
    return [
        {'Season': season, 'DayNum': daynum, 'TeamID': wteam, 'RollingStats': wteam_stats},
        {'Season': season, 'DayNum': daynum, 'TeamID': lteam, 'RollingStats': lteam_stats}
    ]

In [37]:


# Create a list to gather the results.
rolling_stats_results = []

# Use ProcessPoolExecutor to run each game in parallel.
with ProcessPoolExecutor() as executor:
    # Submit each row for processing
    futures = [executor.submit(process_game, row) for _, row in games.iterrows()]
    
    # As each future completes, collect its results.
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Games"):
        results = future.result()
        rolling_stats_results.extend(results)

# Create a DataFrame from the aggregated results.
rolling_stats_df = pd.DataFrame(rolling_stats_results)

# Display the results (using your provided display tool).
tools.display_dataframe_to_user(name="Rolling Stats Data", dataframe=rolling_stats_df)


BrokenProcessPool: A child process terminated abruptly, the process pool is not usable anymore

In [ ]:
rollsing_stats_df.save("rolling_stats_df.csv")

In [46]:
def compute_rolling_stats(long_df, n_matches=5):
    """
    Compute the rolling (previous n_matches) average for each statistic for each team in a season.
    A shift of 1 is applied so that the current game is not included.
    Also computes the rolling averages for one-hot encoded location indicators.
    """
    # Define the stat columns to average
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
                 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    # Create one-hot columns for location
    long_df['home']    = (long_df['Loc'] == 'H').astype(int)
    long_df['away']    = (long_df['Loc'] == 'A').astype(int)
    long_df['neutral'] = (long_df['Loc'] == 'N').astype(int)
    onehot_cols = ['home', 'away', 'neutral']
    
    # Sort by TeamID, Season, and DayNum so the rolling window is correct.
    long_df = long_df.sort_values(by=['TeamID', 'Season', 'DayNum'])
    
    # Compute rolling average for each statistic (shift to use only previous games)
    for col in stat_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    for col in onehot_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    return long_df

In [40]:
pip install dask

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached PyYAML-6.0.2-cp313-cp313-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 7.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [42]:
import dask
from dask import delayed, compute
from dask.diagnostics import ProgressBar
from tqdm.notebook import tqdm
import modin.pandas as pd  # Ensure you're using Modin for multi-core operations

# Assume compute_rolling_stats_cpu is defined as before, using vectorized Modin operations.
def compute_rolling_stats_cpu(match_df, Team_id, Season, DayNum, n_matches=5):
    stat_columns = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
        'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    # Filter matches where the team was involved
    match_df = match_df[(match_df['WTeamID'] == Team_id) | (match_df['LTeamID'] == Team_id)]
    
    # Sort by Season and DayNum
    match_df = match_df.sort_values(['Season', 'DayNum'])
    
    # Filter only matches before the given Season and DayNum
    match_df = match_df[(match_df['Season'] < Season) | ((match_df['Season'] == Season) & (match_df['DayNum'] < DayNum))]
    
    # Take the last n_matches
    last_n_matches = match_df.tail(n_matches).copy()

    # Use vectorized assignment instead of apply
    for col in stat_columns:
        last_n_matches[col] = last_n_matches[f'W{col}'].where(
            last_n_matches['WTeamID'] == Team_id,
            other=last_n_matches[f'L{col}']
        )

    # Compute rolling averages for the selected matches
    rolling_stats = last_n_matches[stat_columns].mean()
    
    # Return as a dictionary (this helps with serialization)
    return {
        'last_matches': last_n_matches.to_dict(orient='list'),
        'rolling_stats': rolling_stats.to_dict()
    }

# Batch size: adjust this based on your dataset size and task duration.
batch_size = 100

# Convert games DataFrame rows into a list so that we can batch them.
rows = list(games.iterrows())
batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]

@delayed
def process_batch(batch):
    batch_results = []
    for _, row in batch:
        season, daynum, wteam, lteam = row['Season'], row['DayNum'], row['WTeamID'], row['LTeamID']
        try:
            wteam_stats = compute_rolling_stats_cpu(df, wteam, season, daynum)
            lteam_stats = compute_rolling_stats_cpu(df, lteam, season, daynum)
            batch_results.extend([
                {'Season': season, 'DayNum': daynum, 'TeamID': wteam, 'RollingStats': wteam_stats},
                {'Season': season, 'DayNum': daynum, 'TeamID': lteam, 'RollingStats': lteam_stats}
            ])
        except Exception as e:
            print(f"Error processing row {row.to_dict()}: {e}")
    return batch_results

# Create a delayed task for each batch.
delayed_batches = [process_batch(batch) for batch in batches]

with ProgressBar():
    batched_results = compute(*delayed_batches, scheduler='processes')

# Flatten the list of results (each batch returns a list of dictionaries).
flat_results = [item for sublist in batched_results for item in sublist]

# Create a DataFrame from the aggregated results.
rolling_stats_df = pd.DataFrame(flat_results)
tools.display_dataframe_to_user(name="Rolling Stats Data", dataframe=rolling_stats_df)


[########################################] | 100% Completed | 301.31 s


ImportError: Please refer to installation documentation page to install an engine

In [44]:
flat_results

[{'Season': np.int64(2003),
  'DayNum': np.int64(10),
  'TeamID': np.int64(1104),
  'RollingStats': {'last_matches': {'Season': [],
    'DayNum': [],
    'WTeamID': [],
    'WScore': [],
    'LTeamID': [],
    'LScore': [],
    'WLoc': [],
    'NumOT': [],
    'WFGM': [],
    'WFGA': [],
    'WFGM3': [],
    'WFGA3': [],
    'WFTM': [],
    'WFTA': [],
    'WOR': [],
    'WDR': [],
    'WAst': [],
    'WTO': [],
    'WStl': [],
    'WBlk': [],
    'WPF': [],
    'LFGM': [],
    'LFGA': [],
    'LFGM3': [],
    'LFGA3': [],
    'LFTM': [],
    'LFTA': [],
    'LOR': [],
    'LDR': [],
    'LAst': [],
    'LTO': [],
    'LStl': [],
    'LBlk': [],
    'LPF': [],
    'Score': [],
    'FGM': [],
    'FGA': [],
    'FGM3': [],
    'FGA3': [],
    'FTM': [],
    'FTA': [],
    'OR': [],
    'DR': [],
    'Ast': [],
    'TO': [],
    'Stl': [],
    'Blk': [],
    'PF': []},
   'rolling_stats': {'Score': nan,
    'FGM': nan,
    'FGA': nan,
    'FGM3': nan,
    'FGA3': nan,
    'FTM': nan,
   